# MC Sim - GPU Build & Test

Build and test the molecular communication GPU simulator on Colab.

**Before running:**
1. Go to Runtime > Change runtime type > T4 GPU
2. Click the 🔑 key icon in the left sidebar, add a secret named `GITHUB_PAT` with your token, and toggle notebook access on

In [ ]:
# Verify GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
# Clone the repo (uses GITHUB_PAT from Colab secrets)
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT')
!git clone https://{PAT}@github.com/alwaysEpic/molecular_modeling_gpu.git
%cd molecular_modeling_gpu

In [ ]:
# Build both targets + RNG dump tool
!mkdir -p build && cd build && cmake .. && make -j$(nproc)
!g++ -O2 -o build/dump_rng_cpu scripts/dump_rng_cpu.cpp -lm
!pip install -q numpy matplotlib scipy
!ls -la build/mc_sim build/mc_sim_cpu build/dump_rng_cpu

## 1. GPU Smoke Test

In [ ]:
!cd build && ./mc_sim -i 1000 -f -v

## 2. CPU vs GPU Comparison

In [ ]:
# 1000 paths
!cd build && ./mc_sim -i 1000 -c -f -v

In [ ]:
# 10000 paths
!cd build && ./mc_sim -i 10000 -c -f -v

## 3. CPU-Only Benchmarks

In [ ]:
!cd build && ./mc_sim_cpu -i 1000 -f -v
print("---")
!cd build && ./mc_sim_cpu -i 1000 -f -v -n

## 4. Analytical Validation — 1D First-Hit with Drift

Compares simulation output against analytical inverse Gaussian (thesis eq 4.3).

In [ ]:
# GPU validation — 1D first-hit with drift (10k paths)
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))

In [ ]:
# CPU validation (1k paths — quick check, not benchmark)
!cd build && ./mc_sim_cpu -i 1000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_h.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))

## 5. Analytical Validation — 3D Spherical Receiver (no drift)

Compares against analytical H_Diff (thesis eq 4.1).

In [ ]:
# GPU - 3D diffusion only
!cd build && ./mc_sim -i 10000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --total-paths 10000
from IPython.display import Image, display
display(Image('validation_3d_diffusion.png'))

In [ ]:
# CPU - 3D diffusion only (1k paths — quick check)
!cd build && ./mc_sim_cpu -i 1000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_h.csv \
    --total-paths 1000
from IPython.display import Image, display
display(Image('validation_3d_diffusion.png'))

## 6. CPU/GPU Agreement (Long and Wide)

KS test comparing hit-time distributions from CPU against both GPU kernel paths.

In [ ]:
# CPU vs GPU Long agreement
print("=== CPU vs GPU Long ===")
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_gpu.csv \
    --timestep 1E-7 --no-plot
print()
# CPU vs GPU Wide agreement
print("=== CPU vs GPU Wide ===")
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 -W > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_gpu.csv \
    --timestep 1E-7 --no-plot

## 7. RNG Quality

Tests statistical moments, autocorrelation, and normality.

In [ ]:
# CPU RNG quality
!cd build && ./dump_rng_cpu 10000 > rng_cpu.csv
!python scripts/validate_rng.py build/rng_cpu.csv --no-plot

In [ ]:
%%writefile /tmp/dump_rng_gpu.cu
// Dump GPU random numbers for RNG quality testing
#include <stdio.h>
#include <stdlib.h>
#include <curand.h>
#include <curand_kernel.h>

__global__ void gen_randn(curandStatePhilox4_32_10_t* rng, float* out, int n, int draws_per_thread) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  for (int d = 0; d < draws_per_thread; d++) {
    out[idx * draws_per_thread + d] = curand_normal(&rng[idx]);
  }
}

__global__ void init_rng(curandStatePhilox4_32_10_t* rng, long long seed, int n) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  curand_init(seed, idx, 0, &rng[idx]);
}

int main(int argc, char** argv) {
  int n_threads = 1000;
  int draws = 10;  // 10 draws per thread = 10000 total
  int total = n_threads * draws;
  long long seed = 42;
  if (argc > 1) seed = atoll(argv[1]);

  curandStatePhilox4_32_10_t* d_rng;
  float* d_out;
  cudaMalloc(&d_rng, n_threads * sizeof(curandStatePhilox4_32_10_t));
  cudaMalloc(&d_out, total * sizeof(float));

  int block = 128;
  int grid = (n_threads + block - 1) / block;
  init_rng<<<grid, block>>>(d_rng, seed, n_threads);
  cudaDeviceSynchronize();
  gen_randn<<<grid, block>>>(d_rng, d_out, n_threads, draws);
  cudaDeviceSynchronize();

  float* out = (float*)malloc(total * sizeof(float));
  cudaMemcpy(out, d_out, total * sizeof(float), cudaMemcpyDeviceToHost);

  for (int i = 0; i < total; i++) printf("%0.15f\n", out[i]);

  free(out);
  cudaFree(d_rng);
  cudaFree(d_out);
  return 0;
}

In [ ]:
# Compile and run GPU RNG dump, then validate
!nvcc -O2 -o build/dump_rng_gpu /tmp/dump_rng_gpu.cu
!cd build && ./dump_rng_gpu 42 > rng_gpu.csv
print("=== GPU RNG (persistent Philox, seed=42) ===")
!python scripts/validate_rng.py build/rng_gpu.csv --no-plot
print()
!cd build && ./dump_rng_gpu 123 > rng_gpu2.csv
print("=== GPU RNG (persistent Philox, seed=123) ===")
!python scripts/validate_rng.py build/rng_gpu2.csv --no-plot

## 8. Reproducibility (--seed)

Same seed should produce identical output.

In [ ]:
# GPU reproducibility
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_gpu.csv run1_gpu.csv
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_gpu.csv run2_gpu.csv
!diff build/run1_gpu.csv build/run2_gpu.csv && echo 'GPU REPRODUCIBILITY: PASS (identical)' || echo 'GPU REPRODUCIBILITY: FAIL (differs)'

In [ ]:
# CPU reproducibility
!cd build && ./mc_sim_cpu -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_h.csv run1_cpu.csv
!cd build && ./mc_sim_cpu -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_h.csv run2_cpu.csv
!diff build/run1_cpu.csv build/run2_cpu.csv && echo 'CPU REPRODUCIBILITY: PASS (identical)' || echo 'CPU REPRODUCIBILITY: FAIL (differs)'

## 9. Performance Regression Test

Builds the current branch and thesis-baseline branch, runs both back-to-back
in the same session, and fails if current is measurably slower.

In [ ]:
# Build thesis-baseline for comparison
import os, subprocess
from google.colab import userdata

if not os.path.isdir('/content/baseline_build'):
    PAT = userdata.get('GITHUB_PAT')
    !git clone https://{PAT}@github.com/alwaysEpic/molecular_modeling_gpu.git /content/baseline_build
    !cd /content/baseline_build && git checkout thesis-baseline
    !cd /content/baseline_build && mkdir -p build && cd build && cmake .. 2>&1 | tail -1 && make -j$(nproc) 2>&1 | tail -1
else:
    print("Baseline already built")

In [ ]:
import subprocess, re, os

def run_and_time(binary_path, args, cwd, runs=3):
    """Run binary multiple times, extract GPU time, return median."""
    times = []
    for _ in range(runs):
        result = subprocess.run(
            [binary_path] + args,
            capture_output=True, text=True, cwd=cwd
        )
        output = result.stdout + result.stderr
        match = re.search(r'GPU code execution time is ([\d.]+)s', output)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

# Paths
current_dir = os.getcwd()
current_bin = os.path.join(current_dir, "build", "mc_sim")
baseline_bin = "/content/baseline_build/build/mc_sim"

tests = [
    ("1k default first-hit", ["-i", "1000", "-f", "-v"]),
    ("10k default first-hit", ["-i", "10000", "-f", "-v"]),
    ("10k 1D limit", ["-i", "10000", "-f", "-l", "3E-7", "-t", "1E-2", "-v"]),
]

print("=== Performance Regression Test (median of 3 runs) ===\n")
print(f"{'Test':<25} {'Baseline':>10} {'Current':>10} {'Change':>10} {'Status':>8}")
print("-" * 70)

all_pass = True
for name, args in tests:
    t_base = run_and_time(baseline_bin, args, "/content/baseline_build/build")
    t_curr = run_and_time(current_bin, args, os.path.join(current_dir, "build"))

    if t_base and t_curr:
        change = (t_curr - t_base) / t_base * 100
        # FAIL if current is >20% slower (allows for Colab noise)
        status = "FAIL" if change > 20 else "PASS"
        if status == "FAIL":
            all_pass = False
        print(f"{name:<25} {t_base:>9.3f}s {t_curr:>9.3f}s {change:>+9.1f}% {status:>8}")
    else:
        print(f"{name:<25} {'ERR':>10} {'ERR':>10} {'—':>10} {'SKIP':>8}")

print()
if all_pass:
    print("PASS — no performance regression detected")
else:
    print("FAIL — current branch is significantly slower than baseline")

## 9b. Thesis vs Current: Wide Kernel Benchmark

Runs the original thesis code (thesis-baseline branch) and the current optimized
wide kernel on the same GPU, same particle counts. Isolates code improvements
from hardware differences.

The thesis had two GPU kernels — "wide" (per-step) and "long" (all timesteps
in one launch). The thesis-baseline branch only included the wide kernel.

In [ ]:
import subprocess, re, os, time

def run_gpu_time(binary_path, args, cwd, runs=3):
    """Run binary multiple times, extract GPU time, return median."""
    times = []
    for _ in range(runs):
        result = subprocess.run(
            [binary_path] + args,
            capture_output=True, text=True, cwd=cwd
        )
        output = result.stdout + result.stderr
        match = re.search(r'GPU code execution time is ([\d.]+)s', output)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

current_dir = os.getcwd()
current_bin = os.path.join(current_dir, "build", "mc_sim")
baseline_bin = "/content/baseline_build/build/mc_sim"

# 1D first-hit with drift (100k timesteps) — matches thesis Table 4.2 test
test_args_1d = ["-f", "-l", "3E-7", "-t", "1E-2", "-v"]

particle_counts = [1000, 5000, 10000]

# Thesis GTX 1070 numbers from Table 4.2 (with drift, first-hit)
thesis_1070 = {1000: 2.27, 5000: 3.46, 10000: 4.81, 50000: 41.52, 100000: 111.98}

print("=" * 90)
print("Thesis vs Current: 1D First-Hit with Drift (b=300nm, 100k steps)")
print("Same GPU, median of 3 runs")
print("=" * 90)
print()
print(f"{'Particles':<12} {'Thesis 1070':>12} {'Thesis GPU':>12} {'Wide GPU':>12} {'Long GPU':>12} {'Code Speedup':>14}")
print(f"{'':.<12} {'(Table 4.2)':>12} {'(baseline)':>12} {'(current)':>12} {'(current)':>12} {'(wide/base)':>14}")
print("-" * 90)

for n in particle_counts:
    args = ["-i", str(n)] + test_args_1d

    # Thesis baseline (original code)
    t_base = run_gpu_time(baseline_bin, args, "/content/baseline_build/build")

    # Current wide kernel
    t_wide = run_gpu_time(current_bin, args + ["-W"], os.path.join(current_dir, "build"))

    # Current long kernel
    t_long = run_gpu_time(current_bin, args, os.path.join(current_dir, "build"))

    t_1070 = thesis_1070.get(n, None)

    code_speedup = f"{t_base/t_wide:.1f}x" if (t_base and t_wide) else "—"

    print(f"{n:<12,} {t_1070 or 0:>11.3f}s {t_base or 0:>11.3f}s {t_wide or 0:>11.3f}s {t_long or 0:>11.3f}s {code_speedup:>14}")

print()
print("Code Speedup = thesis-baseline / current wide (same GPU)")
print("  (isolates code improvements from hardware differences)")

### Performance Charts (thesis Figures 4.4 / 4.5)

Speed comparison and speedup multiplier, matching the thesis chart style.
Uses data collected from section 9b above.

In [ ]:
import subprocess, re, os
import matplotlib.pyplot as plt
import numpy as np

def run_gpu_time_chart(binary_path, args, cwd, runs=3):
    times = []
    for _ in range(runs):
        result = subprocess.run([binary_path] + args, capture_output=True, text=True, cwd=cwd)
        match = re.search(r'GPU code execution time is ([\d.]+)s', result.stdout + result.stderr)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

def run_cpu_time_chart(binary_path, args, cwd, runs=2):
    times = []
    for _ in range(runs):
        result = subprocess.run([binary_path] + args, capture_output=True, text=True, cwd=cwd)
        match = re.search(r'CPU code execution time.*?is ([\d.]+)', result.stdout + result.stderr)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

current_dir = os.getcwd()
current_gpu = os.path.join(current_dir, "build", "mc_sim")
current_cpu = os.path.join(current_dir, "build", "mc_sim_cpu")
baseline_gpu = "/content/baseline_build/build/mc_sim"

test_args = ["-f", "-l", "3E-7", "-t", "1E-2", "-v"]
particle_counts = [100, 500, 1000, 2000, 5000, 10000]

# Collect timing data
cpu_times, thesis_gpu_times, wide_times, long_times = [], [], [], []

print("Collecting timing data...")
for n in particle_counts:
    args = ["-i", str(n)] + test_args
    print(f"  {n} particles...", end=" ", flush=True)

    t_cpu = run_cpu_time_chart(current_cpu, args, os.path.join(current_dir, "build"))
    t_thesis = run_gpu_time_chart(baseline_gpu, args, "/content/baseline_build/build")
    t_wide = run_gpu_time_chart(current_gpu, args + ["-W"], os.path.join(current_dir, "build"))
    t_long = run_gpu_time_chart(current_gpu, args, os.path.join(current_dir, "build"))

    cpu_times.append(t_cpu)
    thesis_gpu_times.append(t_thesis)
    wide_times.append(t_wide)
    long_times.append(t_long)
    print(f"CPU={t_cpu:.3f}s  Thesis={t_thesis:.3f}s  Wide={t_wide:.3f}s  Long={t_long:.3f}s")

# Convert to arrays
N = np.array(particle_counts)
cpu = np.array(cpu_times)
thesis = np.array(thesis_gpu_times)
wide = np.array(wide_times)
long = np.array(long_times)

# --- Figure 4.4 style: Time vs Number of Paths ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(N, cpu, 'b-o', linewidth=2, markersize=6, label='CPU')
ax1.plot(N, thesis, 'r--s', linewidth=2, markersize=6, label='Thesis GPU (wide)')
ax1.plot(N, wide, 'g-^', linewidth=2, markersize=6, label='Wide kernel')
ax1.plot(N, long, 'k-d', linewidth=2, markersize=6, label='Long kernel')
ax1.set_xlabel('Number of Paths', fontsize=12)
ax1.set_ylabel('Time (s)', fontsize=12)
ax1.set_title('Execution Time vs Particle Count\n(1D first-hit, 100k steps)', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# --- Figure 4.5 style: Speedup Multiplier ---
speedup_thesis = cpu / thesis
speedup_wide = cpu / wide
speedup_long = cpu / long

ax2.plot(N, speedup_thesis, 'r--s', linewidth=2, markersize=6, label='Thesis GPU vs CPU')
ax2.plot(N, speedup_wide, 'g-^', linewidth=2, markersize=6, label='Wide kernel vs CPU')
ax2.plot(N, speedup_long, 'k-d', linewidth=2, markersize=6, label='Long kernel vs CPU')
ax2.axhline(y=1, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Number of Paths', fontsize=12)
ax2.set_ylabel('Speedup (CPU time / GPU time)', fontsize=12)
ax2.set_title('GPU Speedup vs Particle Count\n(1D first-hit, 100k steps)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPeak speedup: Long={max(speedup_long):.0f}x  Wide={max(speedup_wide):.0f}x  Thesis={max(speedup_thesis):.1f}x")

## 10. Wall Reflection Validation

Ported from TobyThesisTest_walls.m. Transmitter and receiver positioned near
the vessel wall. Validates that wall reflection does not corrupt diffusion
statistics by comparing against free-space analytical solution.

In [ ]:
# Wall test: transmitter at (0, 7.8um, 0), receiver at (0, 7.8um, 50nm)
# Vessel radius 8um, no drift, 10k paths, 0.4ms duration
!cd build && ./mc_sim -i 10000 -f -w -n \
    --start-y 7.8E-6 \
    --rec-y 7.8E-6 --rec-z 50E-9 \
    -r 8E-6 -t 0.4E-3 -v
print()
!python scripts/validate_3d_walls.py build/output_gpu.csv \
    --total-paths 10000
from IPython.display import Image, display
display(Image('validation_3d_walls.png'))

## 10b. Wide Kernel Validation

Tests the per-step (wide) kernel path with --wide flag.
This is the architecture for future particle interactions.

In [ ]:
# Wide kernel: 1D limit validation (should match long kernel results)
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -W -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2 --no-plot
print()
# Wide kernel: 3D spherical validation
!cd build && ./mc_sim -i 10000 -f -n -W -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --no-plot --total-paths 10000

## 11. Stress Test (1M paths)

High-sample-count validation with maximum statistical power.
KS critical value at 1M paths (alpha=0.001) is ~0.002 — detects
any CDF discrepancy above 0.2%.

In [ ]:
# 1M paths, 1D limit with drift — ultimate KS test
!cd build && ./mc_sim -i 1000000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))

In [ ]:
# 1M paths, 3D spherical receiver — ultimate binomial + KS test
!cd build && ./mc_sim -i 1000000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --total-paths 1000000
from IPython.display import Image, display
display(Image('validation_3d_diffusion.png'))

In [ ]:
# 100k paths, wall reflection stress test
!cd build && ./mc_sim -i 100000 -f -w -n \
    --start-y 7.8E-6 \
    --rec-y 7.8E-6 --rec-z 50E-9 \
    -r 8E-6 -t 0.4E-3 -v
print()
!python scripts/validate_3d_walls.py build/output_gpu.csv \
    --total-paths 100000 --no-plot

## 12. Maximum Scale Test (10M paths)

The thesis maximum was 200k paths. With the long kernel we can
do 50x more. 10M paths gives KS critical value ~0.0006 at alpha=0.001.

In [ ]:
# 10M paths, 1D limit — 50x beyond thesis maximum
import time
print("Starting 10M path simulation...")
t0 = time.time()
!cd build && ./mc_sim -i 10000000 -f -l 3E-7 -t 1E-2 -v
t1 = time.time()
print(f"\nTotal wall time: {t1-t0:.1f}s")
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2
from IPython.display import Image, display
display(Image('validation_1d_firsthit.png'))